# 05 — Initial Load Orchestrator

Runs all four generation notebooks in sequence to fully seed a Fabric Lakehouse.

**Run this notebook once** to populate a fresh Lakehouse, then use `06_incremental_load` for ongoing updates.

> All parameters are forwarded to child notebooks. Change values in the parameters cell below.

In [ ]:
%run ./00_helpers

In [ ]:
# ── Scale & counts ─────────────────────────────────────────────────────
RANDOM_SEED           = 42
SCHEMA_NAME           = "rockline"
NUM_PRODUCTS          = 600
NUM_SUPPLIERS         = 50
NUM_CUSTOMERS         = 3500
TRADE_CUSTOMER_PCT    = 0.75
NUM_EMPLOYEES         = 340

# ── Date range ─────────────────────────────────────────────────────────
HISTORY_START_DATE    = "2022-01-01"
HISTORY_END_DATE      = "2024-12-31"
HUB_STOCK_MULTIPLIER  = 3.0

# ── Transaction settings ────────────────────────────────────────────────
DAILY_ORDER_TARGET    = 45
MAX_ORDER_LINES       = 12
MAX_PO_LINES          = 15
TRADE_DISCOUNT_MAX_PCT = 15.0
VAT_RATE              = 0.20

## Pre-flight Checks

In [ ]:
from datetime import date
assert NUM_PRODUCTS  >= len(SEED_PRODUCTS), "NUM_PRODUCTS must be >= number of seed products"
assert 0.0 < TRADE_CUSTOMER_PCT < 1.0,     "TRADE_CUSTOMER_PCT must be between 0 and 1"
start = date.fromisoformat(HISTORY_START_DATE)
end   = date.fromisoformat(HISTORY_END_DATE)
assert start < end, "HISTORY_START_DATE must be before HISTORY_END_DATE"

days_count = sum(1 for i in range((end - start).days + 1)
                 if (start + __import__("datetime").timedelta(days=i)).weekday() < 5)

print("Pre-flight checks passed")
print(f"  Schema:       {SCHEMA_NAME}")
print(f"  Products:     {NUM_PRODUCTS}")
print(f"  Customers:    {NUM_CUSTOMERS}  ({round(NUM_CUSTOMERS*TRADE_CUSTOMER_PCT)} trade, {round(NUM_CUSTOMERS*(1-TRADE_CUSTOMER_PCT))} private)")
print(f"  Employees:    {NUM_EMPLOYEES}")
print(f"  History:      {HISTORY_START_DATE} to {HISTORY_END_DATE}  ({days_count} business days)")
print(f"  ~Est. orders: {days_count * DAILY_ORDER_TARGET:,}")

## Run All Steps

In [ ]:
def _args(**kwargs):
    return {k: str(v) for k, v in kwargs.items()}

print("\n── Step 1: Master Data ─────────────────────────────────────────────")
mssparkutils.notebook.run("01_master_data", timeout_seconds=900, arguments=_args(
    RANDOM_SEED=RANDOM_SEED,
    SCHEMA_NAME=SCHEMA_NAME,
    NUM_PRODUCTS=NUM_PRODUCTS,
    NUM_SUPPLIERS=NUM_SUPPLIERS,
    NUM_CUSTOMERS=NUM_CUSTOMERS,
    TRADE_CUSTOMER_PCT=TRADE_CUSTOMER_PCT,
    NUM_EMPLOYEES=NUM_EMPLOYEES,
))

print("\n── Step 2: Opening Inventory ───────────────────────────────────────")
mssparkutils.notebook.run("02_inventory", timeout_seconds=600, arguments=_args(
    RANDOM_SEED=RANDOM_SEED,
    SCHEMA_NAME=SCHEMA_NAME,
    HISTORY_START_DATE=HISTORY_START_DATE,
    HUB_STOCK_MULTIPLIER=HUB_STOCK_MULTIPLIER,
))

print("\n── Step 3: Historical Sales ────────────────────────────────────────")
mssparkutils.notebook.run("03_sales", timeout_seconds=1800, arguments=_args(
    RANDOM_SEED=RANDOM_SEED,
    SCHEMA_NAME=SCHEMA_NAME,
    START_DATE=HISTORY_START_DATE,
    END_DATE=HISTORY_END_DATE,
    DAILY_ORDER_TARGET=DAILY_ORDER_TARGET,
    MAX_ORDER_LINES=MAX_ORDER_LINES,
    TRADE_DISCOUNT_MAX_PCT=TRADE_DISCOUNT_MAX_PCT,
    VAT_RATE=VAT_RATE,
    WRITE_MODE="overwrite",
    ORDER_ID_OFFSET="0",
))

print("\n── Step 4: Purchase Orders ─────────────────────────────────────────")
mssparkutils.notebook.run("04_purchasing", timeout_seconds=900, arguments=_args(
    RANDOM_SEED=RANDOM_SEED,
    SCHEMA_NAME=SCHEMA_NAME,
    START_DATE=HISTORY_START_DATE,
    END_DATE=HISTORY_END_DATE,
    VAT_RATE=VAT_RATE,
    MAX_PO_LINES=MAX_PO_LINES,
    WRITE_MODE="overwrite",
    PO_ID_OFFSET="0",
))

## Completion Summary

In [ ]:
print("\n── Final Row Counts ────────────────────────────────────────────────")
for tbl in [
    "dim_branch", "dim_product_category", "dim_supplier",
    "dim_product", "dim_customer", "dim_employee",
    "fact_inventory_snapshot", "fact_inventory_movement",
    "fact_sales_order", "fact_sales_order_line",
    "fact_purchase_order", "fact_purchase_order_line",
]:
    cnt = spark.table(f"{SCHEMA_NAME}.{tbl}").count()
    print(f"  {tbl:<35} {cnt:>8,} rows")

print("\nInitial load complete.")